<a href="https://colab.research.google.com/github/Naganarthanan/RiceGuard-DL/blob/thirishnavi/Model_4_MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

dataset_path = '/content/drive/MyDrive/rice_disease_split'

for root, dirs, files in os.walk(dataset_path):
    print(root, "->", len(files), "files")

/content/drive/MyDrive/rice_disease_split -> 1 files
/content/drive/MyDrive/rice_disease_split/val -> 0 files
/content/drive/MyDrive/rice_disease_split/val/Healthy_Rice_Leaf -> 97 files
/content/drive/MyDrive/rice_disease_split/val/Brown_Spot -> 95 files
/content/drive/MyDrive/rice_disease_split/val/Leaf_scald -> 94 files
/content/drive/MyDrive/rice_disease_split/val/Leaf_Blast -> 92 files
/content/drive/MyDrive/rice_disease_split/val/Sheath_Blight -> 94 files
/content/drive/MyDrive/rice_disease_split/val/Bacterial_Leaf_Blight -> 95 files
/content/drive/MyDrive/rice_disease_split/train -> 0 files
/content/drive/MyDrive/rice_disease_split/train/Bacterial_Leaf_Blight -> 445 files
/content/drive/MyDrive/rice_disease_split/train/Healthy_Rice_Leaf -> 454 files
/content/drive/MyDrive/rice_disease_split/train/Leaf_Blast -> 433 files
/content/drive/MyDrive/rice_disease_split/train/Brown_Spot -> 445 files
/content/drive/MyDrive/rice_disease_split/train/Sheath_Blight -> 439 files
/content/drive/

In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
GPU Available: []


In [4]:
train_dir = '/content/drive/MyDrive/rice_disease_split/train'
val_dir   = '/content/drive/MyDrive/rice_disease_split/val'
test_dir  = '/content/drive/MyDrive/rice_disease_split/test'   # test folder irundha

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [5]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Class names check pannu
print(train_gen.class_indices)

Found 2655 images belonging to 6 classes.
Found 567 images belonging to 6 classes.
{'Bacterial_Leaf_Blight': 0, 'Brown_Spot': 1, 'Healthy_Rice_Leaf': 2, 'Leaf_Blast': 3, 'Leaf_scald': 4, 'Sheath_Blight': 5}


In [6]:
num_classes = train_gen.num_classes

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False   # freeze - initial-ஆ base layers train aagadhu

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,726 (9.24 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint(
        '/content/drive/MyDrive/mobilenetv2_stage1_best.h5',
        save_best_only=True,
        monitor='val_accuracy'
    )
]

In [9]:
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 14s/step - accuracy: 0.4703 - loss: 1.3844 

83/83 ━━━━━━━━━━━━━━━━━━━━ 1523s 18s/step - accuracy: 0.5247 - loss: 1.2494 - val_accuracy: 0.6737 - val_loss: 0.9016
Epoch 2/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6139 - loss: 1.0363

83/83 ━━━━━━━━━━━━━━━━━━━━ 105s 1s/step - accuracy: 0.6139 - loss: 1.0342 - val_accuracy: 0.6966 - val_loss: 0.8154
Epoch 3/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 991ms/step - accuracy: 0.6655 - loss: 0.8680

83/83 ━━━━━━━━━━━━━━━━━━━━ 136s 1s/step - accuracy: 0.6637 - loss: 0.8855 - val_accuracy: 0.7213 - val_loss: 0.7353
Epoch 4/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 990ms/step - accuracy: 0.6755 - loss: 0.8413

83/83 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.6802 - loss: 0.8430 - val_accuracy: 0.7425 - val_loss: 0.6852
Epoch 5/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7035 - loss: 0.7775

...

83/83 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.7307 - loss: 0.7166 - val_accuracy: 0.7901 - val_loss: 0.5745
Epoch 11/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7457 - loss: 0.6617

83/83 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.7446 - loss: 0.6696 - val_accuracy: 0.7919 - val_loss: 0.5557
Epoch 12/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.7540 - loss: 0.6568 - val_accuracy: 0.7901 - val_loss: 0.5518
Epoch 13/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7795 - loss: 0.6155

83/83 ━━━━━━━━━━━━━━━━━━━━ 107s 1s/step - accuracy: 0.7669 - loss: 0.6239 - val_accuracy: 0.8148 - val_loss: 0.5356
Epoch 14/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7615 - loss: 0.6406

83/83 ━━━━━━━━━━━━━━━━━━━━ 136s 1s/step - accuracy: 0.7529 - loss: 0.6501 - val_accuracy: 0.8201 - val_loss: 0.5345
Epoch 15/15
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 997ms/step - accuracy: 0.7446 - loss: 0.6641 

83/83 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.7563 - loss: 0.6442 - val_accuracy: 0.8307 - val_loss: 0.4931


In [1]:
import os
import tensorflow as tf

print("="*60)
print("✅ STEP 1: ENVIRONMENT CHECK")
print("="*60)
print("TensorFlow version:", tf.__version__)
gpu = tf.config.list_physical_devices('GPU')
print("GPU Available:", "YES ✅" if gpu else "NO ❌ (Runtime > Change runtime type > GPU)")

print("\n" + "="*60)
print("✅ STEP 2: DATASET STRUCTURE CHECK")
print("="*60)

dataset_path = '/content/drive/MyDrive/rice_disease_split'
expected_classes = ['Bacterial_Leaf_Blight', 'Brown_Spot', 'Healthy_Rice_Leaf',
                     'Leaf_Blast', 'Leaf_scald', 'Sheath_Blight']

for split in ['train', 'val', 'test']:
    split_path = os.path.join(dataset_path, split)
    if not os.path.exists(split_path):
        print(f"❌ {split} folder MISSING at {split_path}")
        continue
    print(f"\n📁 {split.upper()}:")
    total = 0
    for cls in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            count = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))])
            total += count
            status = "✅" if count > 0 else "❌ EMPTY"
            print(f"   {cls:<25} -> {count} images  {status}")
    print(f"   TOTAL: {total} images")

print("\n" + "="*60)
print("✅ STEP 3: DATA GENERATOR CHECK")
print("="*60)
try:
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

    test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
    check_gen = test_datagen.flow_from_directory(
        os.path.join(dataset_path, 'train'),
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical'
    )
    print("✅ Generator working. Classes found:", check_gen.class_indices)
    print("✅ Number of classes:", check_gen.num_classes)

    # sample batch check
    x_batch, y_batch = next(check_gen)
    print("✅ Sample batch image shape:", x_batch.shape, "(should be [batch,224,224,3])")
    print("✅ Sample batch label shape:", y_batch.shape)
    print("✅ Pixel value range:", x_batch.min(), "to", x_batch.max(), "(MobileNetV2 preprocess = -1 to 1)")
except Exception as e:
    print("❌ Generator ERROR:", e)

print("\n" + "="*60)
print("✅ STEP 4: MODEL CHECK")
print("="*60)
try:
    if 'model' in globals():
        print("✅ Model exists in memory")
        print("   Total layers:", len(model.layers))
        print("   Total params:", f"{model.count_params():,}")
        trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
        print("   Trainable params:", f"{trainable:,}")
        print("   Output shape:", model.output_shape, "(should match num_classes)")
    else:
        print("❌ 'model' variable not found — run the model-building cell first")
except Exception as e:
    print("❌ Model check ERROR:", e)

print("\n" + "="*60)
print("✅ STEP 5: TRAINING HISTORY CHECK")
print("="*60)
try:
    if 'history1' in globals():
        h = history1.history
        print("✅ Stage 1 training completed")
        print(f"   Epochs run: {len(h['loss'])}")
        print(f"   Final train accuracy: {h['accuracy'][-1]:.4f}")
        print(f"   Final val accuracy:   {h['val_accuracy'][-1]:.4f}")
        print(f"   Final train loss: {h['loss'][-1]:.4f}")
        print(f"   Final val loss:   {h['val_loss'][-1]:.4f}")
        gap = h['accuracy'][-1] - h['val_accuracy'][-1]
        if gap > 0.15:
            print("   ⚠️  Warning: large gap between train/val accuracy -> possible overfitting")
        else:
            print("   ✅ Train/val gap looks healthy")
    else:
        print("❌ 'history1' not found — run training cell first")
except Exception as e:
    print("❌ History check ERROR:", e)

print("\n" + "="*60)
print("✅ STEP 6: SAVED MODEL FILE CHECK")
print("="*60)
model_file = '/content/drive/MyDrive/mobilenetv2_stage1_best.h5'
if os.path.exists(model_file):
    size_mb = os.path.getsize(model_file) / (1024*1024)
    print(f"✅ Saved model found: {model_file} ({size_mb:.2f} MB)")
else:
    print("❌ Saved model NOT found — check ModelCheckpoint path/callback")

print("\n" + "="*60)
print("🎯 VERIFICATION COMPLETE")
print("="*60)

✅ STEP 1: ENVIRONMENT CHECK
TensorFlow version: 2.20.0
GPU Available: NO ❌ (Runtime > Change runtime type > GPU)

✅ STEP 2: DATASET STRUCTURE CHECK
❌ train folder MISSING at /content/drive/MyDrive/rice_disease_split/train
❌ val folder MISSING at /content/drive/MyDrive/rice_disease_split/val
❌ test folder MISSING at /content/drive/MyDrive/rice_disease_split/test

✅ STEP 3: DATA GENERATOR CHECK
❌ Generator ERROR: [Errno 2] No such file or directory: '/content/drive/MyDrive/rice_disease_split/train'

✅ STEP 4: MODEL CHECK
❌ 'model' variable not found — run the model-building cell first

✅ STEP 5: TRAINING HISTORY CHECK
❌ 'history1' not found — run training cell first

✅ STEP 6: SAVED MODEL FILE CHECK
❌ Saved model NOT found — check ModelCheckpoint path/callback

🎯 VERIFICATION COMPLETE
